In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
df = pd.read_csv("results.tsv", sep="\t")
df.index = range(1, len(df) + 1)
print(f"Total experiments: {len(df)}")
print(df)

In [ ]:
keeps = (df["status"] == "keep").sum()
discards = (df["status"] == "discard").sum()
crashes = (df["status"] == "crash").sum()
baseline = df[df["status"] == "keep"]["f1_score"].iloc[0] if keeps > 0 else None
best_row = (
    df[df["status"] == "keep"]
    .sort_values("f1_score", ascending=False)
    .iloc[0]
    if keeps > 0
    else None
)

print(f"Keep:    {keeps}")
print(f"Discard: {discards}")
print(f"Crash:   {crashes}")
if baseline is not None and best_row is not None:
    delta = best_row["f1_score"] - baseline
    print(f"Baseline F1:  {baseline:.6f}")
    print(f"Best F1:      {best_row['f1_score']:.6f}  (commit {best_row['commit']})")
    print(f"Improvement:  +{delta:.6f}  ({delta / baseline * 100:.2f}%)")
    print(f"Description:  {best_row['description']}")
print(f"Total tokens: {df['total_tokens'].sum():,}")

In [ ]:
color_map = {"keep": "#2ecc71", "discard": "#e74c3c", "crash": "#95a5a6"}
colors = df["status"].map(color_map)

fig, ax = plt.subplots(figsize=(12, 5))

ax.scatter(df.index, df["f1_score"], c=colors, s=80, zorder=3)

kept = df[df["status"] == "keep"]
if len(kept) > 1:
    ax.plot(
        kept.index,
        kept["f1_score"],
        color="#2ecc71",
        linewidth=1.5,
        linestyle="--",
        zorder=2,
        alpha=0.7,
    )

ax.set_xlabel("Experiment #")
ax.set_ylabel("F1 Score")
ax.set_title("promptloop — F1 Score by Experiment")
ax.set_xticks(df.index)
ax.set_xticklabels(df["commit"].tolist(), rotation=45, ha="right", fontsize=7)
ax.grid(axis="y", alpha=0.3)

legend_patches = [
    mpatches.Patch(color="#2ecc71", label="keep"),
    mpatches.Patch(color="#e74c3c", label="discard"),
    mpatches.Patch(color="#95a5a6", label="crash"),
]
ax.legend(handles=legend_patches, loc="lower right")

plt.tight_layout()
plt.savefig("progress.png", dpi=150)
plt.show()
print("Saved progress.png")